[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafaellopesdesa/ML_Mandi/blob/main/activities/application/Exercise_2_2b_BkgvsRef_training.ipynb)

# Exercise 2.2b — Background vs reference

HSF-India lectures at IIT Mandi. Adapted from the [ML4HEP-TIFR NSBI tutorial](https://github.com/iris-hep/nsbi-lhc-toolkit/tree/b1d3cb8c704ee7a54c99367cf6578e5244ac2671/workshops/ml4hep_tifr_colab).

Run the setup cell first, then run the cells in order. Use a **CPU runtime**.
Everything is generated on the runtime's local disk: **20,000 signal and 20,000 background events**, with seed 42.
Training uses two hidden layers of 64 neurons and at most 20 epochs. The expected physics yields are unchanged, so the smaller simulated samples give noisier estimates.
The original exercises and worked likelihood implementations are retained.

Local files last for the lifetime of the runtime. See [the activity guide](https://github.com/rafaellopesdesa/ML_Mandi/tree/main/activities/application) for the notebook order and optional model handoff.


In [ ]:
# Run first: local storage only, in Colab or local Jupyter.
import os
import sys
from pathlib import Path
from urllib.request import urlretrieve

REPOSITORY = "rafaellopesdesa/ML_Mandi"
REF = "main"
N_BKG = N_SIG = 20_000
SEED = 42

if "google.colab" in sys.modules:
    WORKDIR = Path("/content/ML_Mandi_application")
elif (Path.cwd() / "setup_local.py").is_file():
    WORKDIR = Path.cwd()
elif (Path.cwd() / "activities/application/setup_local.py").is_file():
    WORKDIR = Path.cwd() / "activities/application"
else:
    WORKDIR = Path.cwd() / "ML_Mandi_application"

WORKDIR = WORKDIR.resolve()
WORKDIR.mkdir(parents=True, exist_ok=True)
helper = WORKDIR / "setup_local.py"
if not helper.is_file():
    urlretrieve(
        f"https://raw.githubusercontent.com/{REPOSITORY}/{REF}/activities/application/setup_local.py",
        helper,
    )
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))

from setup_local import setup
setup(WORKDIR, repository=REPOSITORY, ref=REF, n_bkg=N_BKG, n_sig=N_SIG, seed=SEED)


Density Ratio Training - Background vs Reference
---

In the 2a and 2b notebooks we set up the neural networks that train unbiased, low-variance **density ratios**, used to build the SBI model

$$ \frac{L(\mu)}{\prod_e p_\mathrm{ref}(x_e)} = \prod_{x_e\in \{x\}} \frac{1}{\mu \cdot \lambda_\mathrm{sig} + \lambda_\mathrm{bkg}} \left[\mu \cdot \lambda_\mathrm{sig} \cdot \frac{p_\mathrm{sig}(x_e)}{p_\mathrm{ref}(x_e)} + \lambda_\mathrm{bkg} \cdot \frac{p_\mathrm{bkg}(x_e)}{p_\mathrm{ref}(x_e)}\right]$$

which we then use to build the test statistic in the `Exercise_2_3_parameter_fitting.ipynb` notebook. The two ratios to estimate with neural networks are

$$ R_\psi^\mathrm{sig}(x) \approx \frac{p_\mathrm{sig}(x)}{p_\mathrm{ref}(x)}, \qquad  R_\psi^\mathrm{bkg}(x) \approx \frac{p_\mathrm{bkg}(x)}{p_\mathrm{ref}(x)}. $$

Following the lecture, each ratio is obtained **not by regressing the ratio directly**, but by training a classifier to separate the target sample ($y=1$) from the reference ($y=0$). The network outputs a score $q_\psi(y=1\mid x)$, optimised with the (weight-balanced) binary cross-entropy

$$ \mathcal{L}_\mathrm{BCE}[q_\psi] = -\,\mathbb{E}_{(x,y)\sim p(x,y)}\big[\,y\log q_\psi + (1-y)\log(1-q_\psi)\,\big], $$

and at the optimum the density ratio is recovered from the score as

$$ R_\psi(x) = \frac{q_\psi(y=1\mid x)}{1 - q_\psi(y=1\mid x)} \approx \frac{p(x)}{p_\mathrm{ref}(x)}. $$

In this notebook we use `nsbi-common-utils` to estimate $R_\psi^\mathrm{bkg}(x)$ (background vs. reference); notebook 2a does the same for $R_\psi^\mathrm{sig}(x)$ (signal vs. reference). Because the reference hypothesis contains the background itself, this ratio is bounded by construction (it cannot exceed the inverse of the background mixing fraction in the reference).

In [ ]:
import nsbi_common_utils
import pandas as pd
import numpy as np
import mplhep as hep
import matplotlib.pyplot as plt

The setup cell has generated the samples locally. Keep this relative path.

In [ ]:
BASE_PATH = "./dataframes/"

Load the signal and background datasets

In [ ]:
signal = pd.read_parquet(f"{BASE_PATH}/signal.parquet")
background = pd.read_parquet(f"{BASE_PATH}/background.parquet")

### Train / inference split

The density ratios must be **trained** and the likelihood **evaluated** on *different* events, otherwise the parameter fit inherits the classifier's overfitting. We split each sample with a fixed seed: the training notebooks (2a, 2b) keep the *training* half and the fitting notebook (2.3) keeps the *inference* half.

`SPLIT_SEED` and `TRAIN_FRACTION` **must be identical across all three notebooks** — the shared seed is what guarantees the two halves are disjoint. Change `TRAIN_FRACTION` (or pass `n_train=<int>` to `split_train_inference`) to adjust how many events go to training vs. inference.

In [ ]:
from utils import split_train_inference

SPLIT_SEED = 0
TRAIN_FRACTION = 0.5  # fraction of each sample used for TRAINING

signal, _ = split_train_inference(signal, train_fraction=TRAIN_FRACTION, seed=SPLIT_SEED)
background, _ = split_train_inference(background, train_fraction=TRAIN_FRACTION, seed=SPLIT_SEED)

print(f"Training on {len(signal):,} signal + {len(background):,} background events")

We will use the full high-dimensional feature space to train the density ratios

In [ ]:
training_features = ["x1", "x2", "x3", "x4", "x5"]

The numerator and denominator hypothesis refer to the background and reference hypothesis respectively for this notebook. Remember, the objective is to train a classifier score $q_\psi(y=1\mid x)$ whose odds estimate

$$ R_\psi^\mathrm{bkg}(x) = \frac{q_\psi(y=1\mid x)}{1 - q_\psi(y=1\mid x)} \approx \frac{p_\mathrm{bkg}(x)}{p_\mathrm{ref}(x)} $$

We will choose the reference hypothesis to be a combination of the signal and background so there are no phase space regions with $p_\mathrm{ref}(x) \approx 0$ during training.

The choice of reference is one of the difficult and less well-defined parts of the SBI approach -  it has significant impact on numerical stability.

In [ ]:
numerator_hypothesis = background.astype('float32').copy()
denominator_hypothesis = pd.concat([background, signal], ignore_index=True).astype('float32').copy()

Build a combined training dataset consisting of events from numerator and denominator hypothesis. By using `weights_normed` the binary cross-entropy loss function sees both classes represented equally, allowing us to use the likelihood-ratio trick for density ratio estimation. `train_labels` correspond to the truth labels that go into the binary cross-entropy loss function.

In [ ]:
numerator_hypothesis["weights"] = numerator_hypothesis['weight']
numerator_hypothesis["weights_normed"] = numerator_hypothesis['weight'] / numerator_hypothesis['weight'].sum()
numerator_hypothesis["train_labels"] = 1.0

denominator_hypothesis["weights"] = denominator_hypothesis['weight']
denominator_hypothesis["weights_normed"] = denominator_hypothesis['weight'] / denominator_hypothesis['weight'].sum()
denominator_hypothesis["train_labels"] = 0.0

In [ ]:
N_TRAIN = 30_000  # Small local sample; None = use all events

training_dataframe = pd.concat([numerator_hypothesis, denominator_hypothesis], ignore_index=True)
training_dataframe = training_dataframe.sample(frac=1, random_state=43, ignore_index=True)

if N_TRAIN is not None:
    training_dataframe = training_dataframe.head(N_TRAIN).reset_index(drop=True)
    # Re-normalize per-class weights so each class still sums to 1 after subsetting
    for label_val in [0.0, 1.0]:
        mask = training_dataframe["train_labels"] == label_val
        training_dataframe.loc[mask, "weights_normed"] = (
            training_dataframe.loc[mask, "weight"]
            / training_dataframe.loc[mask, "weight"].sum()
        )

### Neural Network Training

In [ ]:
ensemble_size = 1

In [ ]:
from nsbi_common_utils.training import density_ratio_trainer

trainer = [None for i in range(ensemble_size)]

for ens_index in range(ensemble_size):
    trainer[ens_index] = density_ratio_trainer(
        dataset=training_dataframe,
        weights=training_dataframe["weights_normed"],
        training_labels=training_dataframe["train_labels"],
        features=training_features,
        features_scaling=training_features,
        sample_name=["background", "reference"],
        output_name="",
        path_to_figures="plots_BkgvsRef/",
        path_to_models="models_BkgvsRef/",
    )

### HyperParameters

The quick-run defaults build a small multi-layer perceptron (MLP). These settings keep the exercise fast on a CPU; calibration and closure may improve with more events and training.

Exercise: try changing hyperparameters like hidden_layers or neurons or batch_size and see what results you get - both in terms of diagnostics and final parameter fits.

In [ ]:
for ens_index in range(ensemble_size):
    trainer[ens_index].train(
        hidden_layers=2,
        neurons=64,
        number_of_epochs=20,
        batch_size=512,
        learning_rate=1e-3,
        scalerType="MinMax",
        ensemble_index=ens_index,
        verbose=1,
        holdout_split=0.25, # validation
        validation_split=0.2,
        callback_patience=10,
        callback_factor=0.5,
        rnd_seed=42,
        num_workers=0,
        load_trained_models=False,
        calibration=False,
        type_of_calibration = "histogram",
        recalibrate_output=True,
        num_bins_cal = 20
    )

from setup_local import record_training
record_training("models_BkgvsRef")


Diagnostic Checks
===

While traditionally, a NN observable is judged on the basis of its accuracy - for NSBI we are interested in the quality of the density ratios more than the discrimination power. The latter comes from the perfect modelling of the multi-dimensional likelihood ratios.

To ensure correct modelling, we run two main checks on the training:

- **Calibration closure test**

  The NNs are trained using the binary cross-entropy loss $\mathcal{L}_\mathrm{BCE}[q_\psi]$, which under ideal conditions leads to the classifier score converging to

  $$q_\psi(y=1\mid x) = \frac{p_\text{b}(x)}{p_\text{b}(x)+p_\text{ref}(x)}$$

  that can be converted into the density ratio we desire (likelihood ratio trick):

  $$R_\psi^\mathrm{bkg}(x) = \frac{p_\text{b}(x)}{p_\text{ref}(x)} = \frac{q_\psi(y=1\mid x)}{1-q_\psi(y=1\mid x)}$$

  For the NNs to be well-calibrated, we use the Monte Carlo samples to verify the equality:


  $$\left[\frac{p_b(x)}{p_b(x)+p_\text{ref}(x)}\right]_\text{NN} \sim \left[\frac{\mathcal{N}_b^{I(x\mid q_\psi)}}{\mathcal{N}_b^{I(x\mid q_\psi)}+\mathcal{N}_\text{ref}^{I(x\mid q_\psi)}}\right]_\text{MC}$$

  where we bin the events from $p_b$ and $p_\text{ref}$ MC samples, denoted by $\mathcal{N}_b^{I(x\mid q_\psi)}$ and $\mathcal{N}_\text{ref}^{I(x\mid q_\psi)}$ respectively, where $I(x\mid q_\psi)$ returns the index of the $q_\psi(y=1\mid x)$ bin in which an event $x$ falls.
 

In [ ]:
ens_index_to_plot = 0

In [ ]:
trainer[ens_index_to_plot].make_calib_plots(observable='score', nbins=20, ensemble_index=0)

## Density ratio reweighting closure tests
  
Despite having a well-calibrated output and thus a robust probabilistic interpretation, the trained density ratios might not capture the full multi-dimensional event information $x$. 

The next diagnostic involves verifying the following equality using 1D projections of $x$:

$$R_\psi^\mathrm{bkg}(x) \times p_{ref}(x) = \frac{p_b(x)}{p_{ref}(x)} \times p_{ref}(x) \sim p_b(x)$$

We can do this one-by-one for all the observables used to model the density ratios, and also possibly the observables not used directly in the training but can still be well-estimated due to the NN learning the right physics.

In [ ]:
trainer[ens_index_to_plot].make_reweighted_plots(training_features, "linear", 20)

### Optional: use this trained model in a separate runtime

The next cell makes a small ZIP containing the model, scaler and data signature.
Download it from the Files sidebar (or your local file browser), then upload it
alongside the other training notebook's ZIP before running Exercise 2.3.
Without these ZIPs, Exercise 2.3 trains its own networks with the quick-run defaults.


In [ ]:
from setup_local import export_models
archive = export_models("models_BkgvsRef")

# Optional, in Colab: uncomment these two lines to download the ZIP.
# from google.colab import files
# files.download(str(archive))
